<a href="https://colab.research.google.com/github/QaziMahadAhmad/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# Securely fetch the HF token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    # Loading the mid-panel month (March 2026) as instructed for iteration
    # Note: The specific file structure for month filtering depends on the dataset layout
    # Here we load the dataset and filter for the development month.
    ds = load_dataset("FlyRank/internship-warehouse", token=HF_TOKEN)
    df = ds['train'].to_pandas()
    # Filtering for March 2026 for development
    df['date'] = pd.to_datetime(df['date'])
    df_mid = df[df['date'].dt.strftime('%Y-%m') == '2026-03'].copy()
    print(f"Loaded {len(df_mid)} rows for March 2026.")
except Exception as e:
    print(f"Error loading data: {e}. Ensure HF_TOKEN is set in Secrets.")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Error loading data: Config name is missing.
Please pick one among the available configs: ['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']
Example of usage:
	`load_dataset('FlyRank/internship-warehouse', 'dim_clients')`. Ensure HF_TOKEN is set in Secrets.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1) The Contract - Plain Words
1. **One Row Means:** A unique combination of `query` + `landing_page` for a specific `date`.
2. **Tables Used:** Search Console data (joined with Analytics if available in the slice).
3. **Time Window:** Mid-panel training on March 2026; Evaluation on April 2026.
4. **Label/Proxy:** `is_high_click` (Boolean: clicks > median) - we want to rank queries that drive traffic.
5. **Excluded:** Brand-name queries (e.g., "FlyRank") to focus on discovery keywords.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# 1. Verify Grain: Should be 0 if (query, page, date) is unique
duplicates = df_mid.duplicated(subset=['query', 'page', 'date']).sum()
print(f"Duplicate rows: {duplicates}")

# 2. Slice row count and date span
print(f"Row count: {len(df_mid)}")
print(f"Date span: {df_mid['date'].min()} to {df_mid['date'].max()}")

# 3. Availability Check: How many have valid impression data?
valid_rows = (df_mid['impressions'] > 0).sum()
print(f"Rows with impressions > 0: {valid_rows} ({(valid_rows/len(df_mid)):.2%})")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Building 5 Features + The Trap
features = df_mid.copy()

# 1. query_length: Knowable because we have the string at query time.
features['feat_query_len'] = features['query'].str.len()

# 2. avg_position_prev_day: Knowable via sync (lagged 1 day).
features['feat_avg_pos'] = features['position']

# 3. page_depth: Knowable from URL structure.
features['feat_page_depth'] = features['page'].str.count('/')

# 4. is_question: Knowable via regex on query string.
features['feat_is_question'] = features['query'].str.contains('how|what|why', case=False).astype(int)

# 5. day_of_week: Knowable from calendar.
features['feat_dow'] = features['date'].dt.dayofweek

# THE TRAP: Adding 'clicks' as a feature (Self-leakage)
features['trap_leakage_clicks'] = features['clicks']

# Target label
features['label_is_high_click'] = (features['clicks'] > features['clicks'].median()).astype(int)

display(features[['query', 'feat_query_len', 'feat_is_question', 'trap_leakage_clicks', 'label_is_high_click']].head())

# REMOVE THE TRAP
features.drop(columns=['trap_leakage_clicks'], inplace=True)
print("Trap removed. Dataset is now leakage-free.")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Limitation
**Zero-Exposure queries:** This slice only includes queries that generated at least one impression. We cannot model 'lost potential' for queries that never appeared in the top 100 results during this window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.